# Jet Image Pixel Distribution

This notebook reads raw jet `.dat` files, converts each event into the same CNN input image format used by the Beta-VAE training code, applies event-level `pT` normalization, and inspects the distribution of normalized pixel values.

The main number printed below is the global mean of pixels satisfying `0 < pixel < 0.1`, which estimates the typical value of nonzero dark pixels after normalization.

In [ ]:
from pathlib import Path
import sys

import csv
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
CODE_DIR = PROJECT_ROOT / "code_training"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from jet_image_utils import (
    load_events_from_dat,
    event_to_jet_image,
    event_to_jet_image_no_pid,
    pt_normalize,
    build_pid_channel_map,
    build_pid_lookup,
)

plt.style.use("default")

## Configuration

Edit `DATA_PATHS` if you want to inspect a different raw input file. The defaults use the local CMS sample files when run from the `cnn_vae/` directory.

In [ ]:
DATA_PATHS = [
    PROJECT_ROOT / "../data" / "u_jet_real_sample.dat",
    #PROJECT_ROOT / "data" / "g_jet_CMS_sample.dat",
]

# Use None to process every event. A finite value keeps exploratory runs fast.
MAX_EVENTS_PER_FILE = None

# Match the no-PID CNN-VAE preprocessing settings used in the README.
IMAGE_SIZE = 32
ETA_MAX = 0.5
PHI_MAX = 0.5

# False: single channel with all particle pT combined.
# True: one channel per PID, matching the PID-channel image path.
USE_PID_CHANNELS = False

DARK_LOW = 0.0
DARK_HIGH = 0.1

data_paths = [Path(p).resolve() for p in DATA_PATHS]
missing = [p for p in data_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing input file(s): " + ", ".join(str(p) for p in missing))

data_paths

## Load Raw Events

In [ ]:
events = []
source_names = []

for path in data_paths:
    file_events = load_events_from_dat(path)
    if MAX_EVENTS_PER_FILE is not None:
        file_events = file_events[:MAX_EVENTS_PER_FILE]
    events.extend(file_events)
    source_names.extend([path.name] * len(file_events))
    print(f"{path.name}: loaded {len(file_events):,} events")

if not events:
    raise ValueError("No events were loaded.")

print(f"Total events: {len(events):,}")
print(f"Particle counts per event: min={min(len(ev) for ev in events)}, "
      f"median={np.median([len(ev) for ev in events]):.0f}, "
      f"max={max(len(ev) for ev in events)}")

## Convert Events To CNN Images

Each raw image stores summed particle `pT` in `(phi, eta)` bins. Then `pt_normalize` divides by the event image sum, so every non-empty image has total pixel sum equal to 1.

In [ ]:
eta_bins = phi_bins = IMAGE_SIZE

if USE_PID_CHANNELS:
    pid_channel_map = build_pid_channel_map(events)
    pid_lookup, pid_min = build_pid_lookup(pid_channel_map)
    n_channels = len(pid_channel_map)
    channel_description = f"PID channels ({n_channels} channels)"
else:
    pid_channel_map = None
    pid_lookup = None
    pid_min = None
    n_channels = 1
    channel_description = "no PID, one total-pT channel"

images = np.empty((len(events), n_channels, phi_bins, eta_bins), dtype=np.float32)
raw_sums = np.empty(len(events), dtype=np.float32)
normalized_sums = np.empty(len(events), dtype=np.float32)

for i, ev in enumerate(events):
    if USE_PID_CHANNELS:
        img = event_to_jet_image(
            ev,
            pid_lookup=pid_lookup,
            pid_min=pid_min,
            n_channels=n_channels,
            eta_bins=eta_bins,
            phi_bins=phi_bins,
            eta_max=ETA_MAX,
            phi_max=PHI_MAX,
        )
    else:
        img = event_to_jet_image_no_pid(
            ev,
            eta_bins=eta_bins,
            phi_bins=phi_bins,
            eta_max=ETA_MAX,
            phi_max=PHI_MAX,
        )
    raw_sums[i] = img.sum()
    images[i] = pt_normalize(img)
    normalized_sums[i] = images[i].sum()

print(f"Image mode: {channel_description}")
print(f"Image array shape: {images.shape} = (events, channels, phi_bins, eta_bins)")
print(f"Raw image pT sum: min={raw_sums.min():.6g}, mean={raw_sums.mean():.6g}, max={raw_sums.max():.6g}")
print(f"Normalized image sum: min={normalized_sums.min():.6g}, mean={normalized_sums.mean():.6g}, max={normalized_sums.max():.6g}")

## Dark Nonzero Pixel Statistics

In [ ]:
pixels = images.reshape(-1)
nonzero_pixels = pixels[pixels > 0]
dark_pixels = pixels[(pixels > DARK_LOW) & (pixels < DARK_HIGH)]

summary = {
    "events": len(events),
    "image_shape": tuple(images.shape[1:]),
    "total_pixels": pixels.size,
    "nonzero_pixels": nonzero_pixels.size,
    "dark_nonzero_pixels": dark_pixels.size,
    "nonzero_fraction": nonzero_pixels.size / pixels.size,
    "dark_fraction_of_all_pixels": dark_pixels.size / pixels.size,
    "dark_fraction_of_nonzero_pixels": dark_pixels.size / nonzero_pixels.size if nonzero_pixels.size else np.nan,
    "dark_pixel_mean_global": float(dark_pixels.mean()) if dark_pixels.size else np.nan,
    "dark_pixel_median_global": float(np.median(dark_pixels)) if dark_pixels.size else np.nan,
    "dark_pixel_std_global": float(dark_pixels.std()) if dark_pixels.size else np.nan,
    "nonzero_pixel_mean_global": float(nonzero_pixels.mean()) if nonzero_pixels.size else np.nan,
    "nonzero_pixel_median_global": float(np.median(nonzero_pixels)) if nonzero_pixels.size else np.nan,
}

for key, value in summary.items():
    print(f"{key}: {value}")

In [ ]:
print(f"Mean of normalized pixels with {DARK_LOW} < pixel < {DARK_HIGH}: "
      f"{summary['dark_pixel_mean_global']:.8g}")
print(f"Median of normalized pixels with {DARK_LOW} < pixel < {DARK_HIGH}: "
      f"{summary['dark_pixel_median_global']:.8g}")
print(f"Count: {summary['dark_nonzero_pixels']:,} pixels")
print(f"Fraction among nonzero pixels: {summary['dark_fraction_of_nonzero_pixels']:.3%}")

## Per-Event Dark Pixel Means

This separates the global pixel-level average from the event-by-event average dark-pixel value.

In [ ]:
dark_mask = (images > DARK_LOW) & (images < DARK_HIGH)
dark_counts_by_event = dark_mask.reshape(len(events), -1).sum(axis=1)
dark_sums_by_event = np.where(dark_mask, images, 0.0).reshape(len(events), -1).sum(axis=1)
dark_means_by_event = np.divide(
    dark_sums_by_event,
    dark_counts_by_event,
    out=np.full(len(events), np.nan, dtype=np.float32),
    where=dark_counts_by_event > 0,
)

valid_event_means = dark_means_by_event[np.isfinite(dark_means_by_event)]

event_summary = {
    "events_with_dark_pixels": int(np.isfinite(dark_means_by_event).sum()),
    "events_without_dark_pixels": int((~np.isfinite(dark_means_by_event)).sum()),
    "dark_count_per_event_min": int(dark_counts_by_event.min()),
    "dark_count_per_event_median": float(np.median(dark_counts_by_event)),
    "dark_count_per_event_max": int(dark_counts_by_event.max()),
    "dark_mean_per_event_mean": float(np.mean(valid_event_means)) if valid_event_means.size else np.nan,
    "dark_mean_per_event_median": float(np.median(valid_event_means)) if valid_event_means.size else np.nan,
    "dark_mean_per_event_min": float(np.min(valid_event_means)) if valid_event_means.size else np.nan,
    "dark_mean_per_event_max": float(np.max(valid_event_means)) if valid_event_means.size else np.nan,
}

for key, value in event_summary.items():
    print(f"{key}: {value}")

## Pixel Value Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].hist(nonzero_pixels, bins=100, color="tab:blue", alpha=0.85)
axes[0].axvline(DARK_HIGH, color="black", linestyle="--", linewidth=1.5, label=f"dark cutoff = {DARK_HIGH}")
axes[0].set_title("All nonzero normalized pixels")
axes[0].set_xlabel("pixel value")
axes[0].set_ylabel("count")
axes[0].set_yscale("log")
axes[0].legend()

axes[1].hist(dark_pixels, bins=100, color="tab:green", alpha=0.85)
axes[1].axvline(summary["dark_pixel_mean_global"], color="black", linestyle="--", linewidth=1.5, label="global mean")
axes[1].set_title(f"Dark nonzero pixels: {DARK_LOW} < pixel < {DARK_HIGH}")
axes[1].set_xlabel("pixel value")
axes[1].set_ylabel("count")
axes[1].set_yscale("log")
axes[1].legend()

axes[2].hist(valid_event_means, bins=80, color="tab:orange", alpha=0.85)
axes[2].axvline(np.nanmean(dark_means_by_event), color="black", linestyle="--", linewidth=1.5, label="mean over events")
axes[2].set_title("Per-event mean of dark pixels")
axes[2].set_xlabel("per-event mean")
axes[2].set_ylabel("events")
axes[2].legend()

fig.tight_layout()
plt.show()

## Mean Jet Image

In [ ]:
mean_image = images.mean(axis=0)

if n_channels == 1:
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(mean_image[0], origin="lower", cmap="magma")
    ax.set_title("Mean normalized jet image")
    ax.set_xlabel("eta bin")
    ax.set_ylabel("phi bin")
    fig.colorbar(im, ax=ax, label="mean normalized pixel")
    plt.show()
else:
    channel_sums = mean_image.reshape(n_channels, -1).sum(axis=1)
    top_channels = np.argsort(channel_sums)[-min(6, n_channels):][::-1]
    fig, axes = plt.subplots(1, len(top_channels), figsize=(4 * len(top_channels), 4))
    if len(top_channels) == 1:
        axes = [axes]
    inv_pid_map = {ch: pid for pid, ch in pid_channel_map.items()}
    for ax, ch in zip(axes, top_channels):
        im = ax.imshow(mean_image[ch], origin="lower", cmap="magma")
        ax.set_title(f"channel {ch}, PID {inv_pid_map[ch]}")
        ax.set_xlabel("eta bin")
        ax.set_ylabel("phi bin")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    plt.show()

## Optional: Save Summary

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "results" / "pixel_distribution"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_DIR / "dark_pixel_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["key", "value"])
    for key, value in summary.items():
        writer.writerow([key, value])
    for key, value in event_summary.items():
        writer.writerow([key, value])

with open(OUTPUT_DIR / "dark_pixel_event_stats.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["source", "raw_pt_sum_in_window", "dark_pixel_count", "dark_pixel_mean"])
    for source, raw_sum, count, mean in zip(source_names, raw_sums, dark_counts_by_event, dark_means_by_event):
        writer.writerow([source, float(raw_sum), int(count), float(mean) if np.isfinite(mean) else ""])

print(f"Saved summary files to: {OUTPUT_DIR}")